# Agricultural Planning with PLAN-T

This tutorial demonstrates the FAO OpenEO **PLAN-T** processes for agricultural decision support. PLAN-T integrates climate data with agronomic models to help farmers and planners make informed decisions about maize planting.

The three PLAN-T processes work together as a pipeline:

1. **`maze_varieties`** — Find suitable maize varieties for a location
2. **`planting_recommendation`** — Check if moisture conditions are sufficient for planting
3. **`cs_assessment`** — Assess climate stressor risks for specific varieties and planting dates

**What you will learn:**
- How to query available maize varieties for any location
- How to get planting readiness recommendations based on soil moisture
- How to assess climate stressor impacts on specific varieties
- How to build a complete planting decision workflow

**Prerequisites:**
- Completed the [Getting Started](01_getting_started.ipynb) tutorial
- `openeo`, `matplotlib`, `pandas` packages installed

**Note:** PLAN-T processes currently cover regions in sub-Saharan Africa where the underlying ADAM Platform has data coverage.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/un-fao/openeo-docs/blob/main/tutorials/05_planting_recommendations.ipynb)

In [ ]:
# !pip install openeo matplotlib pandas

## 1. Connect and Authenticate

In [1]:
import openeo
import json

# Connect to the FAO OpenEO backend
connection = openeo.connect("https://data.fao.org/openeo")

print(f"Connected to: {connection.root_url}")
print(f"API version: {connection.capabilities().api_version()}")

Connected to: https://data.fao.org/openeo/
API version: 1.2.0


In [ ]:
# Authenticate via OIDC — this will open a browser window for FAO SSO login
connection.authenticate_oidc(provider_id="keycloak")

# Verify authentication
user_info = connection.describe_account()
print(f"Logged in as: {user_info}")

## 2. Step 1 — Find Available Maize Varieties

The `maze_varieties` process queries the PLAN-T platform to find maize varieties that are suitable for a given geographic location. This is typically the first step in the planning workflow.

In [ ]:
# Location: a farm in Zambia (sub-Saharan Africa)
latitude = -15.4
longitude = 28.3

# Query available maize varieties
varieties_cube = connection.datacube_from_process(
    process_id="maze_varieties",
    latitude=latitude,
    longitude=longitude,
)

varieties_result = json.loads(varieties_cube.save_result(format="JSON").download())
print("Available maize varieties:")
print(json.dumps(varieties_result, indent=2, default=str))

In [ ]:
# Parse available varieties
import pandas as pd

data = varieties_result.get("result", varieties_result) if isinstance(varieties_result, dict) else varieties_result
if isinstance(data, list):
    print(f"\nFound {len(data)} varieties:")
    for v in data:
        print(f"  - {v}")
elif isinstance(data, dict):
    for key, val in data.items():
        print(f"  {key}: {val}")

### Supported Maize Varieties

The PLAN-T system recognizes these maize varieties:

| Variety | Notes |
|---------|-------|
| ZMS721 | Zambia Seed Co. |
| AFR635 | |
| SC513 | Seed Co. |
| DKC8033 | DeKalb |
| PHB30G19 | Pioneer |
| ADV637W | Advanta |
| PAN53 | Pannar |
| ZMS606 | Zambia Seed Co. |
| AFR638 | |

## 3. Step 2 — Check Planting Readiness

The `planting_recommendation` process checks whether soil moisture conditions are sufficient for planting at a given location and date. This helps avoid planting during dry spells.

In [ ]:
# Check planting readiness for different dates
test_dates = ["2024-11-15", "2024-12-01", "2024-12-15", "2025-01-05"]

print(f"Planting recommendations for ({latitude}, {longitude}):\n")
print(f"{'Date':<15} {'Recommendation'}")
print("-" * 45)

planting_results = {}
for date in test_dates:
    rec_cube = connection.datacube_from_process(
        process_id="planting_recommendation",
        planting_date=date,
        latitude=latitude,
        longitude=longitude,
    )
    rec_result = json.loads(rec_cube.save_result(format="JSON").download())
    planting_results[date] = rec_result
    print(f"{date:<15} {json.dumps(rec_result, default=str)[:80]}")

In [ ]:
# Visualize planting window
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

fig, ax = plt.subplots(figsize=(12, 4))

for i, (date, res) in enumerate(planting_results.items()):
    dt = datetime.strptime(date, "%Y-%m-%d")
    data = res.get("result", res) if isinstance(res, dict) else res

    # Determine recommendation status
    is_sufficient = False
    if isinstance(data, dict):
        for val in data.values():
            if isinstance(val, str) and "sufficient" in val.lower():
                is_sufficient = "insufficient" not in val.lower()
                break
    elif isinstance(data, str):
        is_sufficient = "sufficient" in data.lower() and "insufficient" not in data.lower()

    color = "#2ca02c" if is_sufficient else "#d62728"
    label = "Sufficient" if is_sufficient else "Insufficient"
    ax.bar(dt, 1, width=10, color=color, alpha=0.7, edgecolor="white")
    ax.text(dt, 0.5, label, ha="center", va="center", fontsize=9, fontweight="bold", color="white")

ax.set_yticks([])
ax.set_xlabel("Planting Date")
ax.set_title(f"Planting Readiness — Zambia ({latitude}, {longitude})")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b %Y"))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 4. Step 3 — Climate Stressor Assessment

The `cs_assessment` process evaluates how specific climate stressors impact selected maize varieties at a given location and planting date. This is the most detailed analysis in the PLAN-T pipeline.

In [ ]:
# Select varieties and stressors to assess
selected_varieties = ["ZMS721", "SC513", "PAN53"]
climate_stressors = ["drought", "heat", "waterlogging"]
planting_date = "2024-12-01"

# Run the assessment
cs_cube = connection.datacube_from_process(
    process_id="cs_assessment",
    date=planting_date,
    latitude=latitude,
    longitude=longitude,
    maze_varieties=selected_varieties,
    climate_stressors=climate_stressors,
)

cs_result = json.loads(cs_cube.save_result(format="JSON").download())
print("Climate stressor assessment:")
print(json.dumps(cs_result, indent=2, default=str)[:3000])

In [ ]:
# Visualize stressor impact by variety
import numpy as np

data = cs_result.get("result", cs_result) if isinstance(cs_result, dict) else cs_result

fig, ax = plt.subplots(figsize=(10, 6))

if isinstance(data, dict):
    # Try to extract per-variety or per-stressor data
    print("\nAssessment Summary:")
    print(f"{'Key':<30} {'Value'}")
    print("-" * 60)
    for key, val in data.items():
        print(f"{str(key):<30} {str(val)[:60]}")
else:
    print(f"Result type: {type(data)}")
    print(data)

## 5. Complete Planning Workflow

Let's combine all three steps into a complete agricultural planning workflow for a location.

In [ ]:
def run_planning_workflow(connection, lat, lon, planting_date, stressors=None):
    """Run the complete PLAN-T agricultural planning workflow."""
    if stressors is None:
        stressors = ["drought", "heat"]

    print(f"=== Agricultural Planning Report ===")
    print(f"Location: ({lat}, {lon})")
    print(f"Target planting date: {planting_date}")
    print("=" * 40)

    # Step 1: Find varieties
    print("\n1. AVAILABLE MAIZE VARIETIES")
    varieties_cube = connection.datacube_from_process(
        process_id="maze_varieties",
        latitude=lat, longitude=lon,
    )
    varieties = json.loads(varieties_cube.save_result(format="JSON").download())
    print(f"   Result: {json.dumps(varieties, default=str)[:200]}")

    # Step 2: Check planting readiness
    print("\n2. PLANTING READINESS")
    rec_cube = connection.datacube_from_process(
        process_id="planting_recommendation",
        planting_date=planting_date,
        latitude=lat, longitude=lon,
    )
    recommendation = json.loads(rec_cube.save_result(format="JSON").download())
    print(f"   Result: {json.dumps(recommendation, default=str)[:200]}")

    # Step 3: Climate stressor assessment (if varieties are available)
    print("\n3. CLIMATE STRESSOR ASSESSMENT")
    # Use a subset of known valid varieties
    test_varieties = ["ZMS721", "SC513"]
    cs_cube = connection.datacube_from_process(
        process_id="cs_assessment",
        date=planting_date,
        latitude=lat, longitude=lon,
        maze_varieties=test_varieties,
        climate_stressors=stressors,
    )
    assessment = json.loads(cs_cube.save_result(format="JSON").download())
    print(f"   Result: {json.dumps(assessment, default=str)[:200]}")

    return {
        "varieties": varieties,
        "recommendation": recommendation,
        "assessment": assessment,
    }

# Run for our Zambia location
report = run_planning_workflow(
    connection,
    lat=-15.4,
    lon=28.3,
    planting_date="2024-12-01",
    stressors=["drought", "heat"],
)

## 6. Compare Multiple Locations

Let's compare planting readiness across different locations.

In [ ]:
# Define multiple farm locations in sub-Saharan Africa
locations = [
    {"name": "Lusaka, Zambia", "lat": -15.4, "lon": 28.3},
    {"name": "Lilongwe, Malawi", "lat": -13.9, "lon": 33.8},
    {"name": "Harare, Zimbabwe", "lat": -17.8, "lon": 31.0},
]

planting_date = "2024-12-01"

print(f"Planting recommendations for {planting_date}:\n")
print(f"{'Location':<25} {'Recommendation'}")
print("-" * 60)

for loc in locations:
    rec_cube = connection.datacube_from_process(
        process_id="planting_recommendation",
        planting_date=planting_date,
        latitude=loc["lat"],
        longitude=loc["lon"],
    )
    result = json.loads(rec_cube.save_result(format="JSON").download())
    print(f"{loc['name']:<25} {json.dumps(result, default=str)[:60]}")

## Summary

In this tutorial, you learned how to use the PLAN-T agricultural planning processes:

1. **`maze_varieties`** — Query suitable maize varieties for any location
2. **`planting_recommendation`** — Check soil moisture readiness for a planting date
3. **`cs_assessment`** — Assess climate stressor impacts on selected varieties

### Process Reference

| Process | Inputs | Output |
|---------|--------|--------|
| `maze_varieties` | latitude, longitude | List of suitable varieties |
| `planting_recommendation` | planting_date, latitude, longitude | Sufficient/insufficient for planting |
| `cs_assessment` | date, latitude, longitude, maze_varieties, climate_stressors | Stressor risk assessment per variety |

### Previous Tutorials

- **[Getting Started](01_getting_started.ipynb)** — Connection, authentication, first data cube
- **[NDVI Time Series](02_ndvi_time_series.ipynb)** — Vegetation monitoring over time
- **[Zonal Statistics](03_zonal_statistics.ipynb)** — Aggregating data over administrative regions
- **[Climate Risk (CRTB)](04_climate_risk_crtb.ipynb)** — Hazard and vulnerability assessment